<a href="https://colab.research.google.com/github/i-wild/Python_Ejercicios/blob/master/kart_demo_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
// ItemBox.cpp

#include "ItemBox.h"
#include "ArcadeKartMovementComponent.h"
#include "Components/SphereComponent.h"
#include "Components/StaticMeshComponent.h"
#include "TimerManager.h"
#código de juego de carreras tipo Mario Kart, boceto

AItemBox::AItemBox()
{
    PrimaryActorTick.bCanEverTick = false;

    CollisionSphere = CreateDefaultSubobject<USphereComponent>(TEXT("CollisionSphere"));
    CollisionSphere->InitSphereRadius(120.f);
    CollisionSphere->SetCollisionProfileName(TEXT("OverlapAllDynamic"));
    RootComponent = CollisionSphere;

    BoxMesh = CreateDefaultSubobject<UStaticMeshComponent>(TEXT("BoxMesh"));
    BoxMesh->SetupAttachment(RootComponent);
    BoxMesh->SetCollisionEnabled(ECollisionEnabled::NoCollision);

    // Pesos por defecto: ajustar en el Editor según el diseño del juego
    ItemWeights.Add(EKartItemType::Mushroom, 40.f);
    ItemWeights.Add(EKartItemType::Banana, 25.f);
    ItemWeights.Add(EKartItemType::Shell, 25.f);
    ItemWeights.Add(EKartItemType::Star, 10.f);
}

void AItemBox::BeginPlay()
{
    Super::BeginPlay();

    CollisionSphere->OnComponentBeginOverlap.AddDynamic(this, &AItemBox::OnOverlapBegin);
}

void AItemBox::OnOverlapBegin(UPrimitiveComponent* OverlappedComponent, AActor* OtherActor,
    UPrimitiveComponent* OtherComp, int32 OtherBodyIndex, bool bFromSweep, const FHitResult& SweepResult)
{
    if (!OtherActor || OtherActor == this || !BoxMesh->IsVisible())
    {
        return;
    }

    UArcadeKartMovementComponent* KartMovement = OtherActor->FindComponentByClass<UArcadeKartMovementComponent>();
    if (!KartMovement)
    {
        return; // Solo la recoge algo que sea un kart
    }

    const EKartItemType Item = RollRandomItem();
    OnItemGranted(OtherActor, Item);

    // Ejemplo directo: si es un hongo, aplicamos boost inmediatamente.
    // Para Banana/Shell/Star, normalmente delegarías a un "inventario de ítem"
    // del kart en vez de aplicarlo aquí directo.
    if (Item == EKartItemType::Mushroom)
    {
        KartMovement->ApplyItemBoost(400.f, 1.5f);
    }

    SetCollected(true);
    GetWorldTimerManager().SetTimer(RespawnTimerHandle, this, &AItemBox::HandleRespawn, RespawnTime, false);
}

EKartItemType AItemBox::RollRandomItem() const
{
    float TotalWeight = 0.f;
    for (const auto& Pair : ItemWeights)
    {
        TotalWeight += Pair.Value;
    }

    if (TotalWeight <= 0.f)
    {
        return EKartItemType::Mushroom;
    }

    float Roll = FMath::FRandRange(0.f, TotalWeight);
    for (const auto& Pair : ItemWeights)
    {
        Roll -= Pair.Value;
        if (Roll <= 0.f)
        {
            return Pair.Key;
        }
    }

    return EKartItemType::Mushroom;
}

void AItemBox::SetCollected(bool bCollected)
{
    BoxMesh->SetVisibility(!bCollected);
    CollisionSphere->SetCollisionEnabled(bCollected ? ECollisionEnabled::NoCollision : ECollisionEnabled::QueryOnly);
}

void AItemBox::HandleRespawn()
{
    SetCollected(false);
}